# 026-gated-correlation-head analysis

# Imports

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from uavloc.matching.benchmark import paired_frame_bootstrap
from uavloc.track import experiment_dir, load_runs

# Params

In [ ]:
EXP = "026-gated-correlation-head"
REFERENCE_EXP = "025-a-contrario-gate"
REFERENCE_RUN = "025.1"
HEAD_EXP = "023-learned-correlation-head"
BASELINE_RUN = "026.1"
HEAD_RUNS = ["026.2", "026.3"]
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 10.0
BAR_DELTA_M = -0.5
PRIMARY_HARM_MAX = 0.0125         # 4 of 320 draws
REPRODUCE_TOL_M = 0.001
PARITY_TOL_M = 1e-6
BEARING_DECIMALS = 6
N_BOOTSTRAP = 10_000
CI_PCT = 95
SEED = 42

# Data

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
runs = {r["run"]: r for r in load_runs(EXP)}
reference = next(r for r in load_runs(REFERENCE_EXP) if r["run"] == REFERENCE_RUN)
present = [r for r in [BASELINE_RUN, *HEAD_RUNS] if r in runs]
present

In [ ]:
def load_records(run: str, prefix: str = "", kind: str = "records") -> pd.DataFrame:
    return pd.read_csv(artifacts / f"{prefix}{kind}-{run}.csv")

In [ ]:
records = {run: load_records(run) for run in present}
holdout = {run: load_records(run, "holdout_") for run in present}

# Checks

In [ ]:
def reproduces_reference() -> dict[str, float | bool]:
    """Check (c): 026.1's primary and holdout mean and harm @ 10 m against 025.1."""
    ours, ref = runs[BASELINE_RUN]["metrics"], reference["metrics"]
    keys = ["mean_position_error_m_at_10m", "harm_share_at_10m", "holdout_mean_position_error_m_at_10m",
            "holdout_harm_share_at_10m"]
    out = {f"{k}_diff": ours[k] - ref[k] for k in keys}
    out["pass"] = all(abs(v) <= REPRODUCE_TOL_M for v in out.values())
    return out

In [ ]:
reproduction = reproduces_reference()
reproduction

In [ ]:
def merged_draws(a: pd.DataFrame, b: pd.DataFrame) -> pd.DataFrame:
    key = ["frame", "noise_m", "bearing_deg"]
    return a.round({"bearing_deg": BEARING_DECIMALS}).merge(b.round({"bearing_deg": BEARING_DECIMALS}), on=key,
                                                             suffixes=("", "_ref"))

In [ ]:
def head_parity(run: str) -> dict[str, float | bool]:
    """Check (a): the ungated head here equals the 023 run it was loaded from, draw by draw, on both benches."""
    head_run = runs[run]["config"]["head_run"].split("/")[1]
    out = {"head_run": head_run}
    for prefix in ("", "holdout_"):
        old = pd.read_csv(experiment_dir(HEAD_EXP) / "artifacts" / f"{prefix}records-{head_run}.csv")
        m = merged_draws(load_records(run, prefix, "ungated-records"), old)
        out[f"{prefix}matched"] = f"{len(m)}/{len(old)}"
        out[f"{prefix}max_abs_diff_m"] = float((m["error_m"] - m["error_m_ref"]).abs().max())
        out[f"{prefix}pass"] = len(m) == len(old) and out[f"{prefix}max_abs_diff_m"] <= PARITY_TOL_M
    return out

In [ ]:
pd.DataFrame([head_parity(run) for run in HEAD_RUNS if run in runs])

In [ ]:
def coverage_parity(run: str) -> dict[str, float]:
    """Check (b): every draw's coverage is the same under the head as under NCC."""
    out = {}
    for prefix, by_run in (("", records), ("holdout_", holdout)):
        m = merged_draws(by_run[run], by_run[BASELINE_RUN])
        out[f"{prefix}max_abs_coverage_diff"] = float((m["coverage"] - m["coverage_ref"]).abs().max())
    return out

In [ ]:
{run: coverage_parity(run) for run in HEAD_RUNS if run in runs}

# Results

In [ ]:
def run_row(run: str) -> dict:
    m = runs[run]["metrics"]
    return {"run": run, "matcher": runs[run]["config"]["matcher"], "head": runs[run]["config"]["head_run"],
            **{f"@{n:g}": m[f"mean_position_error_m_at_{n:g}m"] for n in NOISE_M},
            "harm": m["harm_share_at_10m"], "accept": m["accept_share_at_10m"], "covered": m["mean_position_error_m_covered"],
            "p90": m["p90_position_error_m_at_10m"],
            "holdout": m["holdout_mean_position_error_m_at_10m"], "holdout_harm": m["holdout_harm_share_at_10m"],
            "holdout_accept": m["holdout_accept_share_at_10m"], "holdout_covered": m["holdout_mean_position_error_m_covered"],
            "ungated": m["ungated_mean_position_error_m"], "ungated_holdout": m["ungated_holdout_mean_position_error_m"],
            "ms": m["ms_per_correction"]}

In [ ]:
table = pd.DataFrame([run_row(run) for run in present]).set_index("run")
table.round(3)

In [ ]:
def gate_values(run: str) -> dict:
    c = runs[run]["config"]
    return {"run": run, **{f"{field}_{place.split('-')[0]}": value for field in ("temperature", "gate_s_max_m", "gate_c_min")
                          for place, value in c[field].items()},
            **{f"holdout_{field}": c[f"holdout_{field}"] for field in ("temperature", "gate_s_max_m", "gate_c_min")}}

In [ ]:
pd.DataFrame([gate_values(run) for run in present]).set_index("run")

In [ ]:
def paired(a: pd.DataFrame, b: pd.DataFrame, noise_m: float, frames: set[str] | None = None) -> dict[str, float]:
    """`a - b` per frame at `noise_m`, frame-bootstrap CI; `frames` limits both to a subset."""
    if frames is not None:
        a, b = a[a["frame"].isin(frames)], b[b["frame"].isin(frames)]
    return paired_frame_bootstrap(a, b, noise_m, N_BOOTSTRAP, CI_PCT, SEED)

In [ ]:
def blind_frames(run_records: pd.DataFrame) -> set[str]:
    return set(run_records.groupby("frame")["coverage"].max().loc[lambda c: c == 0].index)

In [ ]:
subsets = {bench: {"all": None, "covered": set(by_run[BASELINE_RUN]["frame"]) - blind_frames(by_run[BASELINE_RUN])}
           for bench, by_run in (("primary", records), ("holdout", holdout))}
rows = [{"run": run, "bench": bench, "subset": subset, "noise_m": noise_m,
         **paired(by_run[run], by_run[BASELINE_RUN], noise_m, frames)}
        for run in HEAD_RUNS if run in runs
        for bench, by_run in (("primary", records), ("holdout", holdout))
        for subset, frames in subsets[bench].items()
        for noise_m in NOISE_M if subset == "all" or noise_m == PRIMARY_NOISE_M]
deltas = pd.DataFrame(rows)
deltas.round(3)

In [ ]:
def delta(run: str, bench: str, noise_m: float = PRIMARY_NOISE_M, subset: str = "all") -> pd.Series:
    return deltas[(deltas["run"] == run) & (deltas["bench"] == bench) & (deltas["noise_m"] == noise_m)
                  & (deltas["subset"] == subset)].iloc[0]

In [ ]:
def bar_check(run: str) -> dict[str, bool]:
    """The proposal's bar, item by item."""
    m, base = runs[run]["metrics"], runs[BASELINE_RUN]["metrics"]
    hold = delta(run, "holdout")
    return {"holdout_delta": hold["diff_m"] <= BAR_DELTA_M and hold["ci_hi"] < 0,
            "holdout_harm": m["holdout_harm_share_at_10m"] <= base["holdout_harm_share_at_10m"],
            "primary_not_worse": delta(run, "primary")["diff_m"] <= 0,
            "primary_harm": m["harm_share_at_10m"] <= PRIMARY_HARM_MAX}

In [ ]:
bars = pd.DataFrame({run: bar_check(run) for run in HEAD_RUNS if run in runs}).T
bars["all"] = bars.all(axis=1)
bars

In [ ]:
verdict = all(r in bars.index for r in HEAD_RUNS) and bool(bars["all"].all())
verdict

In [ ]:
def village_deltas(run: str) -> pd.DataFrame:
    """Holdout @ 10 m, per village: mean error of the run and of 026.1, accept and harm shares."""
    out = []
    for place, group in holdout[run][holdout[run]["noise_m"] == PRIMARY_NOISE_M].groupby("place"):
        base = holdout[BASELINE_RUN][(holdout[BASELINE_RUN]["noise_m"] == PRIMARY_NOISE_M)
                                     & (holdout[BASELINE_RUN]["place"] == place)]
        out.append({"place": place.split("-")[0], "run_m": group["error_m"].mean(), "ncc_m": base["error_m"].mean(),
                    "diff_m": group["error_m"].mean() - base["error_m"].mean(),
                    "accept": (group["decision"] == "accept").mean(), "ncc_accept": (base["decision"] == "accept").mean(),
                    "harm": group["harm"].mean(), "ncc_harm": base["harm"].mean()})
    return pd.DataFrame(out).set_index("place")

In [ ]:
{run: village_deltas(run).round(3) for run in HEAD_RUNS if run in runs}

# Plots

In [ ]:
def plot_cdfs(by_run: dict[str, pd.DataFrame], title: str) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(6, 4))
    for run, r in by_run.items():
        errors = np.sort(r.loc[r["noise_m"] == PRIMARY_NOISE_M, "error_m"].to_numpy())
        ax.plot(errors, np.linspace(0, 1, len(errors)), label=f"{run} {runs[run]['config']['matcher']}")
    ax.axvline(PRIMARY_NOISE_M, color="k", ls=":", lw=1)
    ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws"); ax.set_title(title)
    ax.legend(); ax.grid(alpha=.3); fig.tight_layout()
    return fig

In [ ]:
def plot_village_deltas(title: str) -> plt.Figure:
    fig, ax = plt.subplots(figsize=(8, 3.5))
    frame = pd.DataFrame({run: village_deltas(run)["diff_m"] for run in HEAD_RUNS if run in runs})
    frame.plot.bar(ax=ax, width=.8)
    ax.axhline(0, color="k", lw=.8); ax.set_ylabel(f"gated head - gated NCC, m @ {PRIMARY_NOISE_M:g}"); ax.set_title(title)
    fig.tight_layout()
    return fig

In [ ]:
figures = {"cdf-holdout": plot_cdfs(holdout, "holdout bench @ 10 m, gated"),
           "cdf-primary": plot_cdfs(records, "primary bench @ 10 m, gated"),
           "villages-holdout": plot_village_deltas("holdout, per village")}

In [ ]:
figures_dir = experiment_dir(EXP) / "figures"
figures_dir.mkdir(exist_ok=True)
for name, fig in figures.items():
    fig.savefig(figures_dir / f"{name}.png", dpi=110)
deltas.to_csv(artifacts / "paired-deltas.csv", index=False)
(artifacts / "analysis.json").write_text(json.dumps(
    {"reproduction": reproduction, "bars": bars.astype(bool).to_dict(orient="index"), "verdict": bool(verdict)},
    indent=1, default=float))
verdict